In [1]:
# install pymsql
%pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Step 1: Import Required Libraries
from sqlalchemy import create_engine
import pandas as pd

In [3]:
# STEP 2: Establish a connection between python and sakila database
engine = create_engine("mysql+pymysql://root:BABYb@22@localhost/sakila")

In [7]:
# STEP 3: write the rentals_month function
def rentals_month(engine, month: int, year: int) -> pd.DataFrame:
    """Retrieves raw rental records for a designated month and year."""
    # Write the SQL query
    query = f"""
    SELECT
    r.rental_id,
    r.rental_date,
    r.inventory_id,
    r.customer_id,
    r.staff_id
    FROM rental AS r
    WHERE MONTH(r.rental_date) = {month}
    AND YEAR(r.rental_date) = {year};
    """
    # Executing the query and loading results directly into a DataFrame
    df_rentals = pd.read_sql(query, con=engine)
    return df_rentals

In [8]:
# STEP 4: Develop the rental_count_month function
def rental_count_month(df_rentalas: pd.DataFrame, month: int, year: int) -> pd.DataFrame:
    """Aggregates rental data by customer_id and dynamically titles the count column."""

    # Grouping by customer_id and counting the number of records per group
    df_counts = df_rentals.groupby("customer_id").size().reset_index(name="rental_count")
    
    # Formatting strings to match the requested pattern
    column_name = f"rentals_{month:02d}_{year}"

    # Renaming the generic count column to the custom dynamic name
    df_counts = df_counts.raname(columns={"rental_count": column_name})

    return df_counts
    

In [10]:
# STEP 5: Create the compare_rentals function
def compare_rentals(df_month1: pd.DataFrame, df_month2: pd.DataFrame) -> pd.DataFrame:
    """Merges two monthly datasets and calculates individual customer rental volume differences."""
    
    # Identifying the custom dynamic column names to perform calculations later
    col_m1 = df_month1.columns[1]
    col_m2 = df_month2.columns[1]
    
    # Performing an outer join on customer_id so no customer data is left out
    df_merged = pd.merge(df_month1, df_month2, on="customer_id", how="outer")
    
    # Replacing NaN values with 0 for customers who didn't rent during a specific month
    df_merged[col_m1] = df_merged[col_m1].fillna(0).astype(int)
    df_merged[col_m2] = df_merged[col_m2].fillna(0).astype(int)
    
    # Creating the derived 'difference' field by subtracting Month 1 from Month 2
    df_merged["difference"] = df_merged[col_m2] - df_merged[col_m1]
    
    return df_merged